In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import speech_recognition as sr
from pydub import AudioSegment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from wordcloud import WordCloud
from collections import Counter
import re

c:\Users\Jonny Villareal\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
#Datos de filtros
fecha_i = '20260501'
fecha_f = '20260531'

mes = 'mayo'

In [3]:
# Ruta de la carpeta que contiene los archivos de actividad de bus zonal
ruta_carpeta = 'Z:/01 base_datos/08 varados FMS'

# Fechas de inicio y fin para el filtro
fecha_inicio = f'{fecha_i}'
fecha_fin = f'{fecha_f}'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):
    if nombre_archivo.endswith('_varados.csv'):
        # Extraer la fecha del nombre del archivo
        fecha_archivo = nombre_archivo[:8]  
        
        # Convertir la fecha a un formato adecuado para comparación
        fecha_archivo_dt = pd.to_datetime(fecha_archivo, format='%Y%m%d', errors='coerce')

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:
            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)
            # Leer el archivo CSV omitiendo la primera fila vacía
            df = pd.read_csv(ruta_archivo, encoding='latin', low_memory=False)
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    notas = pd.concat(dataframes, ignore_index=True)

    # # Guardar el DataFrame consolidado en un nuevo archivo
    # desg_troncal.to_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Indicadores/Indicadores_python/desgl_troncal_ago24.csv', index=False)
else:
    print("No se encontraron archivos para consolidar.")
    
notas.head(3)

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO


In [4]:
# Convertir a texto
notas['Observaciones'] = (
    notas['Observaciones']
    .astype(str)
)

# Filtrar solo observaciones que contengan:
# desvío, desvio, desvios, desvíos, etc.

filtro = notas['Observaciones'].str.contains(
    r'desv[ií]o[s]?',
    case=False,
    na=False,
    regex=True
)

# Dejar solo esos registros
notas_desvios = notas[filtro].copy()

notas_desvios.head()

,Fecha,Concesión,Concesionario de Operación,Tipo de Servicio,Id de línea,Nombre Línea,N° Vehículo,Código Conductor,Nombre Conductor,ID Nota,Tipo Nota,Subtipo Nota,Subsubtipo Nota,Observaciones,Creado Por,Modificado Por
0,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993891,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123251/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
1,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993893,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123260/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
2,1/05/2026 2:09,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10184,740,0,0,,993895,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123262/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
3,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993900,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,\nAutorización desvío/ ID: 123513/ 01-05-2026/...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO
4,1/05/2026 2:13,ENGATIVA ZN,GMOVIL ENGATIVA,URBANO,10196,E25,0,0,,993901,INTERRUPCION DE SERVICIO ZONAL,DESVIO BUS ZONAL,GENERAL,Autorización desvío/ ID: 123516/ 01-05-2026/ 0...,ESTEBAN MEDINA CAMILO,ESTEBAN MEDINA CAMILO


In [5]:
notas_desvios.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Automatización/{mes}_notas.csv', index=False, sep=';')